# allwires_v1 — Complete pipeline: poles, wires, downguys

End-to-end from raw cloud + labeled pole locations to final 4-class LAS. Runs after manual KML labelling.

**Inputs:**
- `laser_left.las` — the 32M-point raw cloud
- `pole_locations.csv` — must have `Attribute` column (run `poles.ipynb` + Google Earth labelling first)

**Outputs:**
- `allwires_v1.las` — 4 classes: POLE, SAME_LANE_WIRE, CROSS_ROAD_WIRE, DOWNGUY
- `allwires_v1.qml` — QGIS style file
- `allwires_v1_spans.csv` — per-wire metadata (pole pair, span, kind, point count)

**Improvements over `complete_and_wires.ipynb`:**
- Wires now extend fully to pole attachments (fixed along-margin + RANSAC catenary fit replacing DBSCAN banding)
- Downguy detection added (diagonal guy wires from pole top to ground anchor)
- Single end-to-end run (no intermediate `poles_no_trees.las` needed)
- Per-pole raw cylinders cached once, reused everywhere — ~3× faster

In [1]:
import time
import numpy as np
import pandas as pd
import laspy
from scipy.spatial import cKDTree
from sklearn.cluster import KMeans, DBSCAN

# --- paths ---
RAW_LAS  = 'laser_left.las'
POLE_CSV = 'pole_locations.csv'
OUT_LAS  = 'allwires_v1.las'
OUT_QML  = 'allwires_v1.qml'
OUT_CSV  = 'allwires_v1_spans.csv'

# --- class IDs (LAS user-defined range) ---
CLS_POLE       = 65
CLS_SAME_LANE  = 70
CLS_CROSS_ROAD = 71
CLS_DOWNGUY    = 72

## 1. Load raw cloud + pole locations (one-time setup)

Build the KDTree on the raw cloud once — every downstream step queries it.

In [2]:
t0 = time.time()

print('Loading raw LAS...')
raw = laspy.read(RAW_LAS)
raw_pts = np.vstack([raw.x, raw.y, raw.z]).T
print(f'  {len(raw_pts):,} raw points')

print('Building KDTree...')
raw_tree = cKDTree(raw_pts[:, :2])

print('Loading pole locations...')
df = pd.read_csv(POLE_CSV)
# Inject user's manual labels from backup latlon CSV
if 'Attribute' not in df.columns:
    raise ValueError('pole_locations.csv has no Attribute column — do the Google Earth labelling first')
df['Attribute'] = df['Attribute'].astype(str).str.strip()

poles = df[df['Attribute'] != 'Tree'].reset_index(drop=True).copy()
print(f'  {len(df)} total clusters, {len(poles)} kept (non-trees)')
print(f'  Breakdown: {dict(poles["Attribute"].value_counts())}')

print(f'\nSetup done in {time.time()-t0:.1f}s')

Loading raw LAS...


  31,905,423 raw points
Building KDTree...


Loading pole locations...
  64 total clusters, 37 kept (non-trees)
  Breakdown: {'Electrical Pole': np.int64(33), 'Streetlight': np.int64(2), 'Pole': np.int64(2)}

Setup done in 29.9s


## 2. Cache per-pole raw cylinders + compute robust tops

For each pole, grab raw points within 12m XY once. This cache is reused for:
- Pole body extraction (tight 0.5m / wide 2.0m cylinder)
- Base completion (down to ground)
- Downguy detection (diagonal features out to 10m)

Compute robust pole tops (98th percentile Z in a tight cylinder) and ground Z (5th percentile in a wide radius).

In [3]:
t0 = time.time()

R_CACHE     = 12.0    # cache radius per pole (enough for wires + downguys)
R_SHAFT     = 0.5
R_TOP       = 2.0
TOP_ZONE    = 1.5     # height below top that is crossarm zone (wide radius)
Z_ABOVE_TOP = 0.5     # include points slightly above top (insulators)

# Per-pole cache: {pid: {'idx': global_raw_indices, 'pts': Nx3, 'top_z', 'ground_z', 'px', 'py'}}
cache = {}

pole_points = []    # final pole body points (Nx3)
pole_pids   = []    # pid per point (for user_data in LAS output)

for _, row in poles.iterrows():
    pid = int(row['Pole_ID'])
    px, py = float(row['X']), float(row['Y'])

    idx = np.array(raw_tree.query_ball_point([px, py], r=R_CACHE), dtype=np.int64)
    if len(idx) < 20:
        print(f'  WARNING: pole {pid} has only {len(idx)} points in cache radius')
        continue
    cyl_pts = raw_pts[idx]

    # Robust top & ground from the cache
    # For top: look at points within 1.5m XY (pole shaft only, not crossarm)
    dxy_cache = np.linalg.norm(cyl_pts[:, :2] - [px, py], axis=1)
    near_shaft = dxy_cache < 1.5
    if near_shaft.sum() < 10:
        top_z = float(row['Z'])
    else:
        top_z = float(np.percentile(cyl_pts[near_shaft, 2], 98))
    ground_z = float(np.percentile(cyl_pts[:, 2], 5))

    cache[pid] = {'idx': idx, 'pts': cyl_pts, 'top_z': top_z, 'ground_z': ground_z,
                  'px': px, 'py': py, 'dxy': dxy_cache}

    # Extract pole body (variable radius by height-from-top)
    h_from_top = top_z - cyl_pts[:, 2]
    z_ok = (cyl_pts[:, 2] > ground_z) & (cyl_pts[:, 2] < top_z + Z_ABOVE_TOP)
    crossarm_zone = h_from_top < TOP_ZONE
    r_ok = (crossarm_zone & (dxy_cache < R_TOP)) | (~crossarm_zone & (dxy_cache < R_SHAFT))
    keep = z_ok & r_ok

    pole_points.append(cyl_pts[keep])
    pole_pids.extend([pid] * int(keep.sum()))

pole_body_pts = np.vstack(pole_points)
pole_body_pid = np.array(pole_pids, dtype=np.int32)

heights = [cache[p]['top_z'] - cache[p]['ground_z'] for p in cache]
print(f'Cached {len(cache)} poles')
print(f'  Total pole body points: {len(pole_body_pts):,}')
print(f'  Heights: min {min(heights):.1f}m, median {np.median(heights):.1f}m, max {max(heights):.1f}m')
print(f'\nCylinder pass done in {time.time()-t0:.1f}s')

Cached 37 poles
  Total pole body points: 83,754
  Heights: min 7.0m, median 10.0m, max 15.2m

Cylinder pass done in 1.5s


## 3. Split poles into lanes (KMeans on X)

Determines which wires are SAME_LANE vs CROSS_ROAD.

In [4]:
X = poles[['X']].values
km = KMeans(n_clusters=2, random_state=0, n_init=10).fit(X)
lane_labels = km.labels_
# Ensure 0=west (lower X), 1=east
if km.cluster_centers_[0, 0] > km.cluster_centers_[1, 0]:
    lane_labels = 1 - lane_labels
poles['Lane'] = lane_labels
lane_by_pid = {int(r['Pole_ID']): int(r['Lane']) for _, r in poles.iterrows()}

split_x = km.cluster_centers_.mean()
print(f'Lane split at X ≈ {split_x:.1f}')
print(f'  West ({(lane_labels==0).sum()} poles): IDs {sorted(int(p) for p, l in lane_by_pid.items() if l==0)}')
print(f'  East ({(lane_labels==1).sum()} poles): IDs {sorted(int(p) for p, l in lane_by_pid.items() if l==1)}')

D:\Downloadss\Installations\Miniconda3\envs\sahithya_lidar\lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


Lane split at X ≈ 356470.9
  West (19 poles): IDs [0, 1, 3, 4, 5, 6, 9, 10, 11, 12, 13, 14, 15, 16, 19, 22, 24, 25, 27]
  East (18 poles): IDs [17, 23, 28, 29, 30, 31, 32, 33, 36, 37, 38, 40, 41, 42, 43, 44, 45, 46]


## 4. Detect wires (SAME_LANE + CROSS_ROAD) with full-span capture

Three-pass detection with iterative refinement:

- **Pass A** — strict tube + RANSAC parabola fit, capture points close to curve
- **Pass B** — looser tube for sparse spans
- **Pass C** — linear (non-catenary) fallback for very sparse cross-road spans

**Fullness tricks:**
- Along-margin reduced to 0.1m (was 0.5m) — captures wires all the way to attachments
- RANSAC parabola fit handles sparse stretches without DBSCAN min_samples=3 cutoff
- Iterative refinement: initial fit → inliers → refit → widen tolerance → final capture
- Cross-section width gate (<0.4m) rejects any tree volume that fits between two poles

In [5]:
t0 = time.time()

MIN_SPAN, MAX_SPAN = 5.0, 90.0
TUBE_A, CAT_A      = 1.8, 0.8
TUBE_B, CAT_B      = 2.5, 1.3
TUBE_C, Z_LINE_C   = 2.5, 1.5
Z_BELOW, Z_ABOVE   = 3.5, 2.0      # Z_ABOVE bumped — insulators sit above pole top
ALONG_MARGIN       = 0.1           # was 0.5 — wires now reach attachments
MIN_COVERAGE       = 0.10
MAX_ACROSS_STD     = 0.4           # tree rejection
RANSAC_ITERS       = 30

# Pre-stack pole positions for fast pair iteration
pole_ids  = poles['Pole_ID'].values.astype(int)
pole_xys  = poles[['X', 'Y']].values
pole_tops = np.array([cache[int(pid)]['top_z'] for pid in pole_ids])

def ransac_parabola(along, z, n_iters=RANSAC_ITERS, inlier_tol=1.0):
    """RANSAC fit z = a + b*along + c*along². Returns coefs or None."""
    n = len(along)
    if n < 3: return None
    best_coef, best_inliers = None, 0
    rng = np.random.default_rng(42)
    for _ in range(n_iters):
        idx = rng.choice(n, size=3, replace=False)
        A = np.stack([np.ones(3), along[idx], along[idx]**2], axis=1)
        try: coef = np.linalg.solve(A, z[idx])
        except np.linalg.LinAlgError: continue
        z_pred = coef[0] + coef[1]*along + coef[2]*along**2
        inliers = np.abs(z - z_pred) < inlier_tol
        n_in = inliers.sum()
        if n_in > best_inliers:
            best_inliers, best_coef = n_in, coef
    if best_coef is None or best_inliers < max(4, int(0.3*n)): return None
    # Final least-squares refit on inliers
    z_pred = best_coef[0] + best_coef[1]*along + best_coef[2]*along**2
    inliers = np.abs(z - z_pred) < inlier_tol
    A = np.stack([np.ones(inliers.sum()), along[inliers], along[inliers]**2], axis=1)
    return np.linalg.lstsq(A, z[inliers], rcond=None)[0]

def project_to_span(i, j, tube_w, aZ, bZ, span_r):
    """Grab raw points near a pole-pair span, project onto along/across axes."""
    A, B = pole_xys[i], pole_xys[j]
    vec = B - A; span = float(np.linalg.norm(vec))
    u = vec / span; n = np.array([-u[1], u[0]])
    cand = np.array(raw_tree.query_ball_point((A+B)/2, r=span/2 + tube_w + 2), dtype=np.int64)
    if len(cand) == 0: return None
    cpts = raw_pts[cand]
    rel = cpts[:, :2] - A
    along = rel @ u; across = rel @ n
    low, high = min(aZ, bZ), max(aZ, bZ)
    in_tube = ((along > ALONG_MARGIN) & (along < span - ALONG_MARGIN) &
               (np.abs(across) < tube_w) &
               (cpts[:, 2] > low - Z_BELOW) & (cpts[:, 2] < high + Z_ABOVE))
    return {'cand': cand, 'cpts': cpts, 'along': along, 'across': across,
            'in_tube': in_tube, 'span': span, 'aZ': aZ, 'bZ': bZ, 'tube_w': tube_w}

def trace_catenary(i, j, tube_w, cat_r):
    """Catenary wire tracing with RANSAC + iterative refinement. Returns list of raw indices."""
    aZ = pole_tops[i]; bZ = pole_tops[j]
    proj = project_to_span(i, j, tube_w, aZ, bZ, None)
    if proj is None: return []
    init = proj['in_tube']
    if init.sum() < 4: return []

    along0 = proj['along'][init]
    z0     = proj['cpts'][init, 2]
    # Cross-section gate on initial points
    if np.std(proj['across'][init]) > MAX_ACROSS_STD * 2: return []

    # RANSAC parabola fit
    coef = ransac_parabola(along0, z0, inlier_tol=cat_r)
    if coef is None: return []

    # Iteration: recapture → refit → widen → final recapture
    span = proj['span']
    along = proj['along']; across = proj['across']; cpts = proj['cpts']
    in_tube = proj['in_tube']

    for iter_r in (cat_r, cat_r * 1.4):
        z_pred = coef[0] + coef[1]*along + coef[2]*along**2
        near = np.abs(cpts[:, 2] - z_pred) < iter_r
        fm = in_tube & near
        if fm.sum() < 4: return []
        # Cross-section gate on fit points
        if np.std(across[fm]) > MAX_ACROSS_STD: return []
        # Coverage check
        ra = along[fm]
        if (ra.max() - ra.min()) / span < MIN_COVERAGE: return []
        # Refit with all inliers
        A_ = np.stack([np.ones(fm.sum()), along[fm], along[fm]**2], axis=1)
        coef = np.linalg.lstsq(A_, cpts[fm, 2], rcond=None)[0]

    z_pred = coef[0] + coef[1]*along + coef[2]*along**2
    final = in_tube & (np.abs(cpts[:, 2] - z_pred) < cat_r * 1.4)
    return [int(g) for g in proj['cand'][final]]

def trace_linear(i, j):
    """Linear-tube fallback for sparse spans (no catenary fit)."""
    aZ = pole_tops[i]; bZ = pole_tops[j]
    proj = project_to_span(i, j, TUBE_C, aZ, bZ, None)
    if proj is None: return []
    cpts = proj['cpts']; along = proj['along']; across = proj['across']
    span = proj['span']
    line_z = aZ + (bZ - aZ) * (along / span)
    mask = (proj['in_tube'] & (np.abs(cpts[:, 2] - line_z) < Z_LINE_C))
    if mask.sum() < 5: return []
    if np.std(across[mask]) > MAX_ACROSS_STD: return []
    if (along[mask].max() - along[mask].min()) / span < 0.05: return []
    return [int(g) for g in proj['cand'][mask]]

# Enumerate pairs within span range (vectorized)
n = len(pole_xys)
di = pole_xys[:, None, :] - pole_xys[None, :, :]
dist_mat = np.linalg.norm(di, axis=2)
ii, jj = np.where((dist_mat > MIN_SPAN) & (dist_mat < MAX_SPAN) & (np.arange(n)[:, None] < np.arange(n)[None, :]))
pairs = list(zip(ii.tolist(), jj.tolist()))
print(f'Candidate pairs: {len(pairs)}')

# Run detection
wire_assignments = {}
span_rows = []
for i, j in pairs:
    pa, pb = int(pole_ids[i]), int(pole_ids[j])
    cross = lane_by_pid[pa] != lane_by_pid[pb]
    cls = CLS_CROSS_ROAD if cross else CLS_SAME_LANE
    kind = 'CROSS_ROAD' if cross else 'SAME_LANE'
    span = float(dist_mat[i, j])

    pts_idx = trace_catenary(i, j, TUBE_A, CAT_A); pass_used = 'A' if pts_idx else None
    if not pts_idx:
        pts_idx = trace_catenary(i, j, TUBE_B, CAT_B); pass_used = 'B' if pts_idx else None
    if not pts_idx:
        pts_idx = trace_linear(i, j); pass_used = 'C' if pts_idx else None

    if pts_idx:
        for g in pts_idx:
            if g not in wire_assignments:
                wire_assignments[g] = cls
        span_rows.append({'pole_a': pa, 'pole_b': pb, 'span_m': round(span, 1),
                          'kind': kind, 'pass': pass_used, 'points': len(pts_idx)})

spans_df = pd.DataFrame(span_rows)
print(f'\nSpans detected: {len(spans_df)}')
if len(spans_df):
    print(f"  SAME_LANE  : {(spans_df['kind']=='SAME_LANE').sum()}")
    print(f"  CROSS_ROAD : {(spans_df['kind']=='CROSS_ROAD').sum()}")
    print(f"  By pass    : {dict(spans_df['pass'].value_counts())}")
print(f'Wire points: {len(wire_assignments):,}')
print(f'\nWire detection done in {time.time()-t0:.1f}s')

Candidate pairs: 103



Spans detected: 32
  SAME_LANE  : 25
  CROSS_ROAD : 7
  By pass    : {'A': np.int64(28), 'C': np.int64(2), 'B': np.int64(2)}
Wire points: 13,390

Wire detection done in 81.0s


## 5. Downguy detection (new class)

Downguys are diagonal cables from pole top to a ground anchor, 3-10m away from the pole base.

**Detection per pole:**
1. Use cached cylinder points, filter to (a) outside 1.5m pole cylinder, (b) between ground+0.5m and top-0.5m
2. For each remaining point, compute unit direction vector from pole top
3. DBSCAN on 3D direction vectors — points along the same downguy cluster by direction
4. Per cluster: check linearity (PCA), Z-range (must span most of pole height), diagonal slope
5. Label surviving clusters as DOWNGUY

In [6]:
t0 = time.time()

R_DOWNGUY_MIN   = 1.5     # exclude pole cylinder itself
R_DOWNGUY_MAX   = 10.0    # typical anchor distance
DG_DIR_EPS      = 0.10    # DBSCAN tolerance on normalized direction vectors
DG_MIN_SAMPLES  = 5
DG_LINEARITY    = 0.92    # PCA linearity threshold
DG_CROSS_STD    = 0.25    # max cross-section width (tight — guys are thin cables)
DG_Z_SPAN_FRAC  = 0.45    # cluster must span this fraction of pole height
DG_SLOPE_MIN    = 0.25    # dz/dxy — minimum diagonal slope (not too horizontal)

downguy_assignments = set()
downguy_rows = []

for pid, c in cache.items():
    pts  = c['pts']
    idx  = c['idx']
    dxy  = c['dxy']
    top_z, ground_z = c['top_z'], c['ground_z']
    px, py = c['px'], c['py']

    # Filter candidates: outside pole cylinder, within max radius, between ground+0.5 and top-0.5
    mask = ((dxy > R_DOWNGUY_MIN) & (dxy < R_DOWNGUY_MAX) &
            (pts[:, 2] > ground_z + 0.5) & (pts[:, 2] < top_z - 0.5))
    if mask.sum() < 20: continue
    cpts = pts[mask]
    local_idx = np.where(mask)[0]
    # --- Memory Fix: Decimate if too many candidates (Antigravity Patch) ---
    MAX_DBSCAN_PTS = 5000
    if len(cpts) > MAX_DBSCAN_PTS:
        step = len(cpts) // MAX_DBSCAN_PTS
        cpts = cpts[::step]
        local_idx = local_idx[::step]
        print(f'  Pole {pid}: Decimating {mask.sum():,} -> {len(cpts):,} pts')

    # Direction vector from pole top, normalized
    top = np.array([px, py, top_z])
    vec = cpts - top
    norms = np.linalg.norm(vec, axis=1)
    unit = vec / norms[:, None]

    # DBSCAN on direction vectors — groups points along the same downguy
    if len(unit) < DG_MIN_SAMPLES: continue
    labels = DBSCAN(eps=DG_DIR_EPS, min_samples=DG_MIN_SAMPLES).fit(unit).labels_

    for lbl in set(labels):
        if lbl == -1: continue
        cluster = labels == lbl
        if cluster.sum() < DG_MIN_SAMPLES: continue
        cp = cpts[cluster]

        # PCA linearity
        ctr = cp.mean(axis=0)
        cov = np.cov((cp - ctr).T) + np.eye(3)*1e-6
        evals = np.sort(np.linalg.eigvalsh(cov))[::-1]
        linearity = (evals[0] - evals[1]) / evals[0] if evals[0] > 0 else 0
        if linearity < DG_LINEARITY: continue

        # Z-span: must reach from near ground to near top
        z_span = cp[:, 2].max() - cp[:, 2].min()
        pole_h = top_z - ground_z
        if z_span / pole_h < DG_Z_SPAN_FRAC: continue

        # Diagonal slope check
        cp_dxy = np.linalg.norm(cp[:, :2] - [px, py], axis=1)
        # Fit z = a + b*dxy; slope b should be negative-ish (going down as we go out)
        A_ = np.stack([np.ones_like(cp_dxy), cp_dxy], axis=1)
        coef, *_ = np.linalg.lstsq(A_, cp[:, 2], rcond=None)
        slope = coef[1]
        if abs(slope) < DG_SLOPE_MIN: continue   # too horizontal

        # Cross-section tightness: perpendicular distances to the fitted 3D line
        # Use the principal PCA direction as the line
        evecs = np.linalg.eigh(cov)[1]
        principal = evecs[:, np.argmax(np.linalg.eigvalsh(cov))]
        rel = cp - ctr
        proj = rel @ principal
        perp = rel - np.outer(proj, principal)
        cross_std = np.std(np.linalg.norm(perp, axis=1))
        if cross_std > DG_CROSS_STD: continue

        # Passes all checks — mark as downguy
        global_idx = idx[local_idx[cluster]]
        downguy_assignments.update(int(g) for g in global_idx)
        downguy_rows.append({'pole_id': pid, 'n_points': int(cluster.sum()),
                             'slope': round(float(slope), 2), 'z_span_m': round(float(z_span), 1),
                             'linearity': round(float(linearity), 3)})

dg_df = pd.DataFrame(downguy_rows)
print(f'Downguys detected on {len(dg_df["pole_id"].unique()) if len(dg_df) else 0} poles ({len(dg_df)} total downguys)')
print(f'Downguy points: {len(downguy_assignments):,}')
if len(dg_df):
    print(dg_df.head(15).to_string(index=False))
print(f'\nDownguy detection done in {time.time()-t0:.1f}s')

  Pole 0: Decimating 16,634 -> 5,545 pts


  Pole 1: Decimating 176,057 -> 5,031 pts
  Pole 4: Decimating 9,642 -> 9,642 pts


  Pole 5: Decimating 8,539 -> 8,539 pts


  Pole 6: Decimating 70,962 -> 5,069 pts


  Pole 10: Decimating 9,901 -> 9,901 pts


...

  Pole 43: Decimating 149,880 -> 5,169 pts


  Pole 44: Decimating 7,897 -> 7,897 pts


  Pole 45: Decimating 66,333 -> 5,103 pts


  Pole 46: Decimating 180,838 -> 5,024 pts


Downguys detected on 4 poles (5 total downguys)
Downguy points: 1,213
 pole_id  n_points  slope  z_span_m  linearity
       4      1106  -1.67       8.8      0.996
       9        22  -3.05       6.5      0.997
      22        10  -1.42       7.0      0.998
      22        14  -2.00       5.2      0.995
      36        61  -0.84       5.6      0.997

Downguy detection done in 9.9s


## 6. Save final LAS + QML

Four classes. Wires (blue/red) win over Pole (green) where they overlap; downguys (orange) win over everything (last-write).

In [7]:
t0 = time.time()

# Remove any raw indices that appear in both wires and downguys (downguy wins — more specific)
for g in downguy_assignments:
    wire_assignments.pop(g, None)

# Assemble final point arrays
if wire_assignments:
    wg = np.array(list(wire_assignments.keys()), dtype=np.int64)
    wire_pts = raw_pts[wg]
    wire_cls = np.array([wire_assignments[int(g)] for g in wg], dtype=np.uint8)
else:
    wire_pts = np.zeros((0, 3)); wire_cls = np.zeros(0, dtype=np.uint8)

if downguy_assignments:
    dg = np.array(list(downguy_assignments), dtype=np.int64)
    downguy_pts = raw_pts[dg]
    downguy_cls = np.full(len(dg), CLS_DOWNGUY, dtype=np.uint8)
else:
    downguy_pts = np.zeros((0, 3)); downguy_cls = np.zeros(0, dtype=np.uint8)

final_pts = np.vstack([pole_body_pts, wire_pts, downguy_pts])
final_cls = np.concatenate([
    np.full(len(pole_body_pts), CLS_POLE, dtype=np.uint8),
    wire_cls,
    downguy_cls,
])
final_pid = np.clip(np.concatenate([
    pole_body_pid,
    np.zeros(len(wire_pts), dtype=np.int32),
    np.zeros(len(downguy_pts), dtype=np.int32),
]) + 1, 0, 255).astype(np.uint8)

# Write LAS
header = laspy.LasHeader(point_format=raw.header.point_format, version=raw.header.version)
header.scales  = raw.header.scales
header.offsets = raw.header.offsets
out = laspy.LasData(header)
out.x, out.y, out.z = final_pts.T
out.classification  = final_cls
out.user_data       = final_pid
out.write(OUT_LAS)

# QML
qml = '''<!DOCTYPE qgis PUBLIC 'http://mrcc.com/qgis.dtd' 'SYSTEM'>
<qgis styleCategories="Symbology" version="3.34">
  <renderer-v2 type="pointcloudclassified" attribute="Classification">
    <categories>
      <category value="65" label="POLE"            color="60,200,60,255"   render="true"/>
      <category value="70" label="SAME_LANE_WIRE"  color="60,130,220,255"  render="true"/>
      <category value="71" label="CROSS_ROAD_WIRE" color="220,60,60,255"   render="true"/>
      <category value="72" label="DOWNGUY"         color="255,140,0,255"   render="true"/>
    </categories>
  </renderer-v2>
</qgis>
'''
with open(OUT_QML, 'w') as f: f.write(qml)

# Spans CSV (wires only; downguys summarized above)
if len(spans_df):
    spans_df.sort_values(['kind','pole_a','pole_b']).to_csv(OUT_CSV, index=False)

print(f'Wrote {OUT_LAS} — {len(final_pts):,} points total')
print(f'  POLE           : {(final_cls == CLS_POLE).sum():>7,}')
print(f'  SAME_LANE_WIRE : {(final_cls == CLS_SAME_LANE).sum():>7,}')
print(f'  CROSS_ROAD_WIRE: {(final_cls == CLS_CROSS_ROAD).sum():>7,}')
print(f'  DOWNGUY        : {(final_cls == CLS_DOWNGUY).sum():>7,}')
print(f'Wrote {OUT_QML}')
if len(spans_df): print(f'Wrote {OUT_CSV}')
print(f'\nSave done in {time.time()-t0:.1f}s')

Wrote allwires_v1.las — 98,262 points total
  POLE           :  83,754
  SAME_LANE_WIRE :   8,527
  CROSS_ROAD_WIRE:   4,768
  DOWNGUY        :   1,213
Wrote allwires_v1.qml
Wrote allwires_v1_spans.csv

Save done in 0.2s


## Done 🎉

In QGIS: open `allwires_v1.las`, Properties → Symbology → Load Style → `allwires_v1.qml`.

**Expected view:**
- 37 green poles with completed bases
- Blue SAME_LANE wires running along each side of the road (should now reach all the way to pole attachments)
- Red CROSS_ROAD wires spanning between the two lanes
- Orange DOWNGUY wires slanting diagonally from pole tops to ground anchors

**If wires still appear cut off at pole attachments:** the raw LiDAR didn't capture points there (mobile lidar occlusion from the pole body itself). Not fixable in post-processing. But the visible parts should now extend much closer to the poles than before.

**If no downguys show up:** check the `Downguys detected on X poles` print. Zero = your scene has no visible downguys (plausible — not every pole has them) OR the tolerances are too strict. The main tunables are `DG_LINEARITY` (lower → more permissive), `DG_Z_SPAN_FRAC` (lower → shorter guys accepted), and `DG_CROSS_STD` (higher → noisier cables accepted).